# DSFB-DSCD threshold sweep replay

Generate a run directory from the workspace root with:

```bash
cargo run --release -p dsfb-dscd -- --num-events 8192 --tau-steps 201
```

By default, the notebook locates the workspace root from the current working directory
and then uses `<workspace>/output-dsfb-dscd/`.
You can still set `RUN_DIR` manually to any run folder.
The notebook only visualizes CSV outputs written by `dsfb-dscd`; it does not re-implement DSCD logic.


In [ ]:
from pathlib import Path

def discover_output_root() -> Path:
    cwd = Path.cwd().resolve()
    checked = []

    for base in [cwd, *cwd.parents]:
        checked.append(base / 'output-dsfb-dscd')

        # Prefer an existing output directory if present.
        if (base / 'output-dsfb-dscd').is_dir():
            return base / 'output-dsfb-dscd'

        # If this looks like the repo root, return the expected output path there.
        if (base / 'Cargo.toml').exists() and (base / 'crates' / 'dsfb-dscd').exists():
            return base / 'output-dsfb-dscd'

    checked_lines = '\n'.join(f'  - {path}' for path in checked)
    raise FileNotFoundError(
        'Could not locate workspace root/output directory. Checked:\n' + checked_lines
    )

OUTPUT_ROOT = discover_output_root()
RUN_DIR = None  # e.g. OUTPUT_ROOT / '20260302_000000'

if RUN_DIR is None:
    run_candidates = sorted([p for p in OUTPUT_ROOT.glob('*') if p.is_dir()])
    if not run_candidates:
        raise FileNotFoundError(
            f'No DSCD output folders found under {OUTPUT_ROOT}. '
            'Generate one with: cargo run --release -p dsfb-dscd -- --num-events 8192 --tau-steps 201'
        )
    RUN_DIR = run_candidates[-1]

RUN_DIR


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

threshold_path = RUN_DIR / 'threshold_sweep.csv'
if not threshold_path.exists():
    raise FileNotFoundError(
        f'{threshold_path} not found. Confirm RUN_DIR points to a dsfb-dscd run output folder.'
    )

threshold_df = pd.read_csv(threshold_path)
threshold_df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_df['tau'], threshold_df['expansion_ratio'], linewidth=2)
ax.set_title('DSCD expansion ratio vs trust threshold')
ax.set_xlabel('tau')
ax.set_ylabel('expansion_ratio')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
finite_size_path = RUN_DIR / 'finite_size_scaling.csv'
if finite_size_path.exists():
    finite_df = pd.read_csv(finite_size_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(finite_df['num_events'], finite_df['transition_width'], marker='o')
    axes[0].set_title('Transition width vs N')
    axes[0].set_xlabel('num_events')
    axes[0].set_ylabel('transition_width')
    axes[0].grid(alpha=0.25)

    axes[1].plot(finite_df['num_events'], finite_df['max_derivative'], marker='o')
    axes[1].set_title('Max derivative vs N')
    axes[1].set_xlabel('num_events')
    axes[1].set_ylabel('max_derivative')
    axes[1].grid(alpha=0.25)
    plt.show()
else:
    print('finite_size_scaling.csv not present for this run')

In [ ]:
events_path = RUN_DIR / 'graph_events.csv'
edges_path = RUN_DIR / 'graph_edges.csv'

events_df = pd.read_csv(events_path)
edges_df = pd.read_csv(edges_path)

subset_edges = edges_df.head(128)
subset_nodes = sorted(set(subset_edges['from_event_id']).union(subset_edges['to_event_id']))

graph = nx.DiGraph()
graph.add_nodes_from(subset_nodes)
graph.add_edges_from(subset_edges[['from_event_id', 'to_event_id']].itertuples(index=False, name=None))

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(graph, seed=7)
nx.draw_networkx(graph, pos=pos, node_size=120, with_labels=False, arrows=True)
plt.title('DSCD graph preview')
plt.axis('off')
plt.show()